图像卷积

在卷积层中，输入张量和核张量通过(互相关运算)产生输出张量。

二维互相关运算。阴影部分是第一个输出元素，以及用于计算输出的输入张量元素和核张量元素：$0\times0+1\times1+3\times2+4\times3=19$.
![二维互相关运算。阴影部分是第一个输出元素，以及用于计算输出的输入张量元素和核张量元素：$0\times0+1\times1+3\times2+4\times3=19$.](../img/correlation.svg)

输出大小等于输入大小$n_h \times n_w$减去卷积核大小$k_h \times k_w$，即：
$$(n_h-k_h+1) \times (n_w-k_w+1).$$

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def corr2d(X, K):  #@save
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

验证上述二维互相关运算的输出

In [3]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

卷积层

卷积层对输入和卷积核权重进行互相关运算，并在添加标量偏置之后产生输出。两个被训练的参数是卷积核权重和标量偏置，需要对其进行初始化。

在__init__构造函数中，将weight和bias声明为两个模型参数。前向传播函数调用corr2d函数并添加偏置。

In [4]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

图像中目标的边缘检测

应用：找到像素变化的位置，检测不同颜色的边缘。

首先，我们构造一个$6\times 8$像素的黑白图像。中间四列为黑色（$0$），其余像素为白色（$1$）。


In [5]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

接下来构造一个高度为$1$、宽度为$2$的卷积核`K`，进行互相关运算时，水平相邻的两元素相同，输出为0，否则非0.

In [6]:
K = torch.tensor([[1.0, -1.0]])

In [7]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

输出`Y`中的1代表从白色到黑色的边缘，-1代表从黑色到白色的边缘

若将X转置，再进行互相关运算，检测到的垂直边缘就消失了，可见该卷积核智能检测垂直边缘。

corr2d(X.t(), K)

学习卷积核：学习由X生成Y的卷积核

先构造一个卷积层，并将其卷积核初始化为随机张量。接下来，在每次迭代中，我们比较Y与卷积层输出的平方误差，然后计算梯度来更新卷积核。

In [9]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = nn.Conv2d(1,1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # 学习率

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷积核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 11.562
epoch 4, loss 3.078
epoch 6, loss 0.982
epoch 8, loss 0.356
epoch 10, loss 0.138


查看学习到的权重张量

In [10]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 0.9488, -1.0240]])

## 练习

1. 构建一个具有对角线边缘的图像`X`。
    1. 如果将本节中举例的卷积核`K`应用于`X`，会发生什么情况？
    1. 如果转置`X`会发生什么？
    1. 如果转置`K`会发生什么？


In [26]:
X = torch.tril(torch.ones((6, 6)))
X

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [27]:
Z = corr2d(X, K)
Z

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])

A. 卷积核能够把边缘检测出来

In [28]:
X_T = X.T
X_T

tensor([[1., 1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.]])

In [30]:
Y_T = corr2d(X_T, K)
Y_T

tensor([[ 0.,  0.,  0.,  0.,  0.],
        [-1.,  0.,  0.,  0.,  0.],
        [ 0., -1.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  0.],
        [ 0.,  0.,  0.,  0., -1.]])

转置后，每一行的变化变成了0到1，边缘方向变了，所以边缘变为-1

In [31]:
K_T = K.T
K_T

Y_KT = corr2d(X, K_T)
Y_KT

tensor([[ 0., -1.,  0.,  0.,  0.,  0.],
        [ 0.,  0., -1.,  0.,  0.,  0.],
        [ 0.,  0.,  0., -1.,  0.,  0.],
        [ 0.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  0.,  0.,  0.,  0., -1.]])

转置卷积核，相当于把“看左右变化”改成“看上下变化”。

1. 在我们创建的`Conv2D`自动求导时，有什么错误消息？

In [32]:
Y_hat = conv2d(X)
Y_hat.backward()

RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [6, 6]

只有“标量输出”才能自动创建梯度，Y_hat是一个矩阵

1. 如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？

将每个窗口和卷积核摊平，再进行矩阵乘法

In [33]:
def im2col(X, kernel_size):
    h, w = kernel_size
    out_h = X.shape[0] - h + 1
    out_w = X.shape[1] - w + 1

    cols = []
    for i in range(out_h):
        for j in range(out_w):
            patch = X[i:i+h, j:j+w].reshape(-1)
            cols.append(patch)

    return torch.stack(cols)  # (num_patches, h*w)

In [34]:
def corr2d_matmul(X, K):
    h, w = K.shape

    # 1. 展开输入
    X_col = im2col(X, (h, w))   # (N, h*w)

    # 2. 展开卷积核
    K_flat = K.reshape(-1, 1)   # (h*w, 1)

    # 3. 矩阵乘法
    Y = X_col @ K_flat          # (N, 1)

    # 4. reshape回图像
    out_h = X.shape[0] - h + 1
    out_w = X.shape[1] - w + 1

    return Y.reshape(out_h, out_w)

In [35]:
# 输入
X = torch.tensor([
    [0., 1., 2.],
    [3., 4., 5.],
    [6., 7., 8.]
])

K = torch.tensor([
    [0., 1.],
    [2., 3.]
])

# 原始方法
Y1 = corr2d(X, K)

# 矩阵乘法方法
Y2 = corr2d_matmul(X, K)

print("Y (原始卷积) =\n", Y1)
print("\nY (矩阵乘法) =\n", Y2)

# 验证
print("\n是否相等:", torch.allclose(Y1, Y2))

Y (原始卷积) =
 tensor([[19., 25.],
        [37., 43.]])

Y (矩阵乘法) =
 tensor([[19., 25.],
        [37., 43.]])

是否相等: True


1. 手工设计一些卷积核。
    1. 二阶导数的核的形式是什么？
    1. 积分的核的形式是什么？
    1. 得到$d$次导数的最小核的大小是多少？

A. 
- 一维：
  \[
  [1, -2, 1]
  \]

- 二维（拉普拉斯）：
  \[
  \begin{bmatrix}
  0&1&0\\
  1&-4&1\\
  0&1&0
  \end{bmatrix}
  \]

B. 
- 局部积分（求和）：
  \[
  [1,1,1], \quad 
  \begin{bmatrix}
  1&1&1\\
  1&1&1\\
  1&1&1
  \end{bmatrix}
  \]

- 累积积分：需要长核（非有限最小核）

C. d+1
- 一阶：2  
- 二阶：3  
- 三阶：4  